# NB39 - trimmed trailing scores on the full archive, in event time

NB38 screened trimmed trailing scores against forward Sharpe on the dense-polling regime only
(from 2026-04-01): about four months, 70 overlapping decisions. This notebook runs the same
question on the whole archive from mid-2025, which means running it on WEEKLY data for most of
its length: through 2025 the archive holds about one mark per vault per week, one every two days
in January-March 2026, and a mark on most days from April (this notebook keeps one last mark
per UTC day, cell 2).

Forward-filling weekly marks to a daily grid and then computing daily statistics is where the
gotchas live, so nothing here is computed on a daily grid. Every score uses observed marks
only: event log returns between consecutive marks inside the window, trimming by a FRACTION of
events rounded up (so a weekly vault with 13 events in 90 days loses 2 or 4 of them and a daily
vault with 90 loses 9 or 23), staleness measured in days since the vault's own last mark, and forward
outcomes that must contain marks near the end of the window. The primary horizon is 60 days
because a 30-day window holds about four weekly marks; the 30-day horizon is kept as a
secondary target. The three polling regimes are screened together and separately.

**Focus is forward Sharpe.** Verdict DIAGNOSTIC: a screen, not a result. No vault is selected,
masked or tuned by name. **Headline: on 192 decisions over a year, trailing risk-adjusted
scores are positively associated with the next 60 days' Sharpe - the 180-day Sharpe and Sortino
lead at rho 0.26, and 15 of 16 signals clear the computed simultaneous bound with
60-day blocks, 15 with 90-day blocks and 15 with 180-day blocks - and no trim improvement
is detected.** No block choice is both long enough to cover the 180-day persistence of the
trailing scores and numerous enough for reliable controlled bootstrap inference (180-day blocks
leave about 3 per draw), so the bounds are the computed figures under each block choice, not a
controlled family-wise result; NB38's four-month, 30-day-horizon screen could not resolve any
of this.

**Based on:** [38-research-trimmed-return-screen.ipynb](38-research-trimmed-return-screen.ipynb)
(machinery and its two reviews), [33-research-lead-comparison.ipynb](33-research-lead-comparison.ipynb)
(the archive density table that defines the regimes). Snapshot `vault-prices.parquet`
255,548,076 bytes, sha256 `11e7c5e0103e1012`, last mark 2026-09-16 (cell 2).

## Method

Marks: one per vault per UTC day (the last poll of the day). Events: consecutive marks; event
return = log price ratio; event span = days between them. A candidate at decision T needs, in
the trailing window (T-1-W, T-1] for W in 90 and 180 days, at least 8 event returns (9 marks) and a
mark at or before the window start, its last mark within 14 days of T-1, and a TVL of at least
7,500 USD at that mark. Scores per window: return score
(sum of event returns, annualised over W; raw and with the best 10% and 25% of events removed),
Sharpe score (return score over event volatility sqrt(sum r^2 / W x 365), raw and trimmed the
same way), Sortino, event volatility. Forward outcomes over (T, T + H] for H = 60 (primary) and
30: log return from the mark carried at T to the last mark in the window, event volatility,
event Sharpe, log max drawdown on the mark path; a window needs at least 6 (H = 60) or 4
(H = 30) marks and one within 14 days of its end. Panel: 29,912 candidate-dates, 397
vaults, 192 decisions 2025-07-01 to 2026-07-18, 75.8% of rows under 360 days old;
59,998 candidate-dates dropped as stale or under-marked, 25,200 for TVL, 858
for an unobserved forward window (cell 4).
Event returns per 90-day window, median: 12 in the weekly regime, 31 in
the transition, 89 in the dense regime; marks per 60-day forward window 9 /
60 / 60 (cell 4, decision-date regimes).

Inference as NB38 with one change forced by the horizon: per-date signed Spearman averaged
over dates, one two-way cluster bootstrap of NON-WRAPPING 30-decision (60-day) date
blocks x vault clusters, 500 draws, seed 20260917, shared across every hypothesis,
studentised max-T simultaneous lower bounds over the 16-signal family on the primary target
(critical 2.57), the same with 45-decision (90-day) blocks (critical
2.49) and 90-decision (180-day) blocks (critical 2.79, about
3 blocks per draw), paired trimmed-minus-raw differences on the same draws
with their own family bound, and a foresight-oracle reachability assertion (lower bound
0.996, cell 8). Regime cohorts contain only decisions whose whole 60-day horizon lies
inside the regime (crosses: 60, dense Apr 2026 on: 55, transition Jan-Mar 2026: 15, weekly 2025: 62 decisions).

## Key new insights and what did we learn from this experiment?

**1. On a year of data, trailing risk-adjusted scores are positively associated with the next
60 days' Sharpe.** With 60-day blocks, 15 of 16 signals clear the computed simultaneous lower
bound of zero on forward 60-day Sharpe; with 90-day blocks the same 15, and with 180-day
blocks - the longest trailing window, about 3 blocks per draw - 15 (cell 6). The strongest are
the 180-day Sharpe and Sortino scores: `sharpe180_f10` 0.257 (bounds
0.128 / 0.125 / 0.119 at 60 / 90 / 180-day blocks), `sortino180`
0.255 (0.120 / 0.127 / 0.120), `sharpe180_f00`
0.251 (0.118 / 0.126 / 0.117). The 180-day Sharpe and
Sortino scores correlate with forward RETURN at 0.191 to
0.193 and with 30-day forward Sharpe at 0.212 to
0.224. NB38 saw 0.136 for the 180-day Sharpe on four months at a 30-day horizon and
could not clear a family bound; this screen has three times the decisions and a horizon that
holds enough marks. Going from 60- to 90-day blocks leaves the passing set unchanged and moves
individual bounds in both directions by at most 0.012; going to 180-day blocks is the honest
check against the trailing scores' own persistence, at the cost of about 3 blocks per draw,
and its bounds are read with that in mind.

**2. No trim improvement is detected under a family bound.** The return-score trim at 90 days
lifts the correlation from 0.151 to 0.211 (paired difference
0.060 [-0.009, 0.128], add-one p
0.100 alone, simultaneous bound over the 8 paired comparisons
-0.016); at 180 days the difference is 0.010. As in NB38
the trimmed return score takes on a volatility loading (signed correlation with lower forward
volatility 0.126 raw, 0.504 trimmed) and lands where the raw 180-day
scores already are. The Sharpe-score trim differences are -0.013 and
0.006 at 10%, -0.037 and
-0.031 at 25%, with intervals that include zero (cell 6). The old-cohort
90-day 25% trim reads -0.172 [-0.360,
-0.040] on its own interval (cell 10), one of several cohort comparisons and
exploratory. Nothing here establishes that trimming helps or harms; it establishes that no
improvement was found.

**3. Regime cohorts with the forward outcome contained in the regime.** The forward 60 days lie
inside the regime; the trailing 180-day score of an early dense-regime decision still reads
pre-April marks, so this is outcome-containment, not a wholly within-regime comparison. Weekly
2025 (62 decisions whose whole horizon is in 2025): `sharpe180_f00` 0.281 (bound
0.052), `sortino180` 0.278. The transition cohort has too few contained decisions to screen (15).
Dense April on (55 decisions): `sharpe180_f00` 0.204 (bound -0.005).
(cell 10). The weekly-regime estimate is the largest. Two readings are consistent with that and
this screen cannot separate them: quality persisted more in 2025's universe (which was
99.2% under a year old), or coarse, irregularly observed returns - a weekly mark is a
snapshot, and the return between two of them is a week's interval return - together with
trailing scores that barely change between marks make trailing and forward scores mechanically
more alike than daily marks would.

**4. The association is larger and more widespread in the young panel.** Young vaults
(75.8% of rows, 192 decisions): `sharpe180_f00` 0.264 (bound
0.126), `ret90_f10` 0.225 (bound 0.111). Old vaults
(102 decisions): `sharpe180_f00` 0.193 (bound 0.014),
`sortino180` 0.196 (cell 10). The two are estimates on different samples with different
date coverage, not a test of a difference. The incumbent's 360-day CAGR leg cannot score a
young vault at all; the scores that predict best here need 180 days.

**5. What this says about the incumbent's ranker.** Its Sortino leg looks back 45 days and its
CAGR leg 360; this screen has no 45-day window, so the leg itself is not tested, but the pattern
is that 180-day risk-adjusted scores (0.251-0.255) sit
above 90-day ones (0.201-0.212) and above raw
return scores at either length. That is a lead for a ranker test, not a result: the effect on a
six-name book is what the standing gates measure, and portfolio consequences are not claimed
here.

## Summary of results

Forward 60-day Sharpe screen, all regimes (cell 6): signed Spearman, simultaneous lower bounds
over the 16-signal family with 60-day blocks (critical 2.57) and 90-day blocks (critical
2.49), unadjusted one-sided add-one p; the forward volatility column is signed so positive =
the score's good end had LOWER forward volatility.

| signal | rho fwd60 Sharpe | bound, 60-d blocks | bound, 90-d blocks | p | rho fwd60 return | rho fwd60 vol | rho fwd30 Sharpe |
|---|---|---|---|---|---|---|---|
| ret90_f00 | 0.151 | 0.018 | 0.021 | 0.002 | 0.138 | 0.126 | 0.111 |
| ret90_f10 | 0.211 | 0.091 | 0.093 | 0.002 | 0.199 | 0.504 | 0.179 |
| ret90_f25 | 0.209 | 0.086 | 0.087 | 0.002 | 0.203 | 0.579 | 0.178 |
| sharpe90_f00 | 0.212 | 0.086 | 0.091 | 0.002 | 0.153 | 0.145 | 0.172 |
| sharpe90_f10 | 0.199 | 0.070 | 0.074 | 0.002 | 0.146 | 0.174 | 0.169 |
| sharpe90_f25 | 0.176 | 0.030 | 0.027 | 0.002 | 0.136 | 0.255 | 0.155 |
| sortino90 | 0.201 | 0.077 | 0.085 | 0.002 | 0.147 | 0.151 | 0.158 |
| vol90 | 0.160 | 0.037 | 0.043 | 0.004 | 0.167 | 0.675 | 0.141 |
| ret180_f00 | 0.203 | 0.064 | 0.076 | 0.002 | 0.181 | 0.260 | 0.163 |
| ret180_f10 | 0.213 | 0.077 | 0.079 | 0.002 | 0.201 | 0.522 | 0.179 |
| ret180_f25 | 0.196 | 0.057 | 0.058 | 0.002 | 0.189 | 0.594 | 0.163 |
| sharpe180_f00 | 0.251 | 0.118 | 0.126 | 0.002 | 0.192 | 0.257 | 0.214 |
| sharpe180_f10 | 0.257 | 0.128 | 0.125 | 0.002 | 0.193 | 0.280 | 0.224 |
| sharpe180_f25 | 0.219 | 0.087 | 0.076 | 0.002 | 0.181 | 0.349 | 0.184 |
| sortino180 | 0.255 | 0.120 | 0.127 | 0.002 | 0.191 | 0.271 | 0.212 |
| vol180 | 0.110 | -0.037 | -0.038 | 0.032 | 0.122 | 0.612 | 0.098 |

Paired trimmed-minus-raw on forward 60-day Sharpe (cell 6): per-comparison 95% intervals and
add-one p; simultaneous lower bounds over each 8-comparison family are all below zero.

| | 10% of events removed | 25% of events removed |
|---|---|---|
| ret score, 90 d | 0.060 [-0.009, 0.128] p 0.100 | 0.058 [-0.028, 0.138] p 0.164 |
| ret score, 180 d | 0.010 [-0.053, 0.067] p 0.667 | -0.007 [-0.098, 0.074] p 1.000 |
| sharpe score, 90 d | -0.013 [-0.068, 0.028] p 0.723 | -0.037 [-0.125, 0.024] p 0.459 |
| sharpe score, 180 d | 0.006 [-0.052, 0.058] p 0.802 | -0.031 [-0.128, 0.047] p 0.595 |

Per cohort, `sharpe180_f00` on forward 60-day Sharpe (cell 10): weekly 2025
0.281 (bound 0.052), dense 0.204 (-0.005),
young 0.264 (0.126), old 0.193 (0.014).

## Robustness of results

- Nothing is computed on a daily grid: scores and outcomes are event-time on observed marks
  inside their windows, eligibility needs a mark within 14 days of T-1 and 8 events in the
  window, forward windows need 6 (60 d) or 4 (30 d) marks and one within 14 days of the end
  (cell 4). A weekly vault's forward outcome is a 60-day return over about nine snapshots, and
  its "Sharpe" is a coarse quantity built from interval returns between them.
- The screen is reachable: a foresight oracle clears the 17-signal bound at
  0.996 (cell 8).
- Date blocks are 30 decisions (60 days, the horizon) and do not wrap; 45-decision (90-day)
  and 90-decision (180-day) blocks are run as sensitivities (cell 6). The passing set is
  15 / 15 / 15 across the three. The 180-day blocks match the longest trailing
  window but leave about 3 blocks per draw, so no block choice here is both long enough for
  the dependence and numerous enough for a well-behaved bootstrap; the bounds are the computed
  figures under each choice and are not claimed as controlled family-wise evidence.
  192 decisions over about a year hold roughly six non-overlapping 60-day horizons.
- One bootstrap per screen; the regime and cohort screens are separate samples with separate
  bootstraps and their intervals are not comparable across screens in a paired sense. Regime
  cohorts contain only decisions whose whole horizon lies inside the regime.
- Trimmed scores are ranking transformations, not investable returns; forward max drawdown is
  in log units.
- The panel is the archive, not the incumbent's candidate pool; no portfolio claim is made.


## Part 0. Archive, provenance, constants, regimes


In [ ]:
import hashlib, json, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

ARCHIVE = Path.home() / ".cache/tradingstrategy/vaults/downloads/vault-prices.parquet"
raw_bytes = ARCHIVE.read_bytes()
PROVENANCE = {"file": str(ARCHIVE), "bytes": len(raw_bytes), "sha256": hashlib.sha256(raw_bytes).hexdigest()}
del raw_bytes
HYPERCORE_CHAIN = 9999

WINDOWS = (90, 180)
TRIM_FRACTIONS = (0.0, 0.10, 0.25)
HORIZONS = {60: 6, 30: 4}          # forward horizon days -> minimum observed marks in the window
PRIMARY_H = 60
PANEL_START = pd.Timestamp("2025-07-01")
DECISION_STEP_DAYS = 2
MIN_TVL_USD = 7_500.0
MIN_EVENTS = 8        # events strictly inside the window, i.e. at least 9 marks
STALE_DAYS = 14
MIN_CANDIDATES = 8
MIN_DATES = 40
YOUNG_DAYS = 360
DRAWS = 500
#: Date blocks must be at least as long as the forward horizon (60 days = 30 decisions) so that
#: overlapping outcomes stay together inside a block; blocks are NOT wrapped circularly, because
#: joining July 2026 to July 2025 would splice two polling regimes. 45 decisions (90 days) is an
#: intermediate sensitivity; the 180-day score persistence is covered only by DATE_BLOCK_LONG.
DATE_BLOCK = 30
DATE_BLOCK_SENSITIVITY = 45
#: 90 decisions = 180 days, the longest trailing-score window. With 192 decisions this leaves
#: about two blocks per draw, so its bounds are as much a statement about the sample size as
#: about the dependence; it is run because no shorter block covers the score persistence.
DATE_BLOCK_LONG = 90
SEED = 20260917
LEVEL = 0.95
REGIMES = [("weekly 2025", pd.Timestamp("2025-01-01"), pd.Timestamp("2026-01-01")),
           ("transition Jan-Mar 2026", pd.Timestamp("2026-01-01"), pd.Timestamp("2026-04-01")),
           ("dense Apr 2026 on", pd.Timestamp("2026-04-01"), pd.Timestamp("2027-01-01"))]


def regime_of(t):
    """Regime of the decision date."""
    for name, a, b in REGIMES:
        if a <= t < b:
            return name
    return "other"


def regime_contained(t, H):
    """Regime that contains BOTH the decision date and its whole forward horizon, else 'crosses'."""
    for name, a, b in REGIMES:
        if a <= t and t + pd.Timedelta(days=H) < b:
            return name
    return "crosses"


df = pd.read_parquet(ARCHIVE, columns=["address", "chain", "share_price", "total_assets", "name"])
df = df[df["chain"] == HYPERCORE_CHAIN].reset_index()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df[df["timestamp"] >= pd.Timestamp("2025-01-01")].sort_values(["address", "timestamp"])
df["date"] = df["timestamp"].dt.floor("D")
marks = df.groupby(["address", "date"])[["share_price", "total_assets"]].last().reset_index()
marks = marks[marks["share_price"] > 0]
names = df.groupby("address")["name"].last()
LAST_MARK = df["timestamp"].max()
display(pd.Series({**PROVENANCE, "last_mark": str(LAST_MARK), "hypercore_vaults": marks["address"].nunique(),
                   "mark_days": len(marks)}, name="value").to_frame())

# Polling density by month on the mark-day grid: marks per vault per day among vaults above the TVL floor.
mm = marks[marks["total_assets"] >= MIN_TVL_USD].copy()
mm["month"] = mm["date"].dt.to_period("M")
dens = mm.groupby("month").agg(vaults=("address", "nunique"), mark_days=("date", "size"))
dens["mark_days_per_vault_per_day"] = dens["mark_days"] / dens["vaults"] / 30.0
display(dens.round(3).T)


## Part 1. The event-time panel

One row per (decision date, vault). Nothing is forward-filled: a window's statistics come from
the marks inside it, and a vault whose last mark is older than 14 days is not a candidate.


In [ ]:
def event_scores(r: np.ndarray, W: int) -> dict:
    """Raw and trimmed return and Sharpe scores from event returns `r` in a window of W days.
    Trimming removes the ceil(f x n) largest event returns; the window length stays the denominator,
    so these are ranking scores, not investable returns."""
    out = {}
    n = len(r)
    order = np.sort(r)
    for f in TRIM_FRACTIONS:
        k = int(math.ceil(f * n)) if f > 0 else 0
        kept = order[:n - k] if k else order
        rate = float(kept.sum() * 365.0 / W)
        vol = float(math.sqrt((kept ** 2).sum() * 365.0 / W)) if len(kept) else float("nan")
        tag = f"f{int(f * 100):02d}"
        out[f"ret_{tag}"] = rate
        out[f"sharpe_{tag}"] = rate / vol if vol and vol > 0 else float("nan")
    downside = math.sqrt((np.clip(r, None, 0.0) ** 2).sum() * 365.0 / W)
    out["sortino"] = out["ret_f00"] / downside if downside > 0 else float("nan")
    out["vol"] = float(math.sqrt((r ** 2).sum() * 365.0 / W))
    out["events"] = int(n)
    return out


def forward_outcome(carried: float, fwd_prices: np.ndarray, H: int) -> dict:
    """Outcome from the mark carried at T to the marks in (T, T + H]."""
    prices = np.concatenate([[carried], fwd_prices])
    r = np.diff(np.log(prices))
    total = float(r.sum())
    vol = float(math.sqrt((r ** 2).sum() * 365.0 / H))
    path = np.concatenate([[0.0], np.cumsum(r)])
    return {f"fwd{H}_return": total, f"fwd{H}_vol": vol,
            f"fwd{H}_sharpe": (total * 365.0 / H) / vol if vol > 0 else float("nan"),
            f"fwd{H}_log_max_dd": float(np.min(path - np.maximum.accumulate(path))),
            f"fwd{H}_events": int(len(fwd_prices))}


max_h = max(HORIZONS)
last_decision = LAST_MARK.floor("D") - pd.Timedelta(days=max_h)
decisions = pd.date_range(PANEL_START, last_decision, freq=f"{DECISION_STEP_DAYS}D")
rows = []
dropped = {"stale_or_too_few_marks": 0, "tvl": 0, "forward_marks": 0}
for address, g in marks.groupby("address"):
    g = g.set_index("date").sort_index()
    mdays = g.index
    prices = g["share_price"].to_numpy()
    tvls = g["total_assets"].to_numpy()
    born = mdays[0]
    for t in decisions:
        t1 = t - pd.Timedelta(days=1)
        i_last = int(mdays.searchsorted(t1, side="right")) - 1   # last mark at or before T-1
        if i_last < 0 or (t1 - mdays[i_last]).days > STALE_DAYS:
            dropped["stale_or_too_few_marks"] += 1
            continue
        if tvls[i_last] < MIN_TVL_USD:
            dropped["tvl"] += 1
            continue
        row = {"address": address, "date": t, "age_days": int((t - born).days), "regime": regime_of(t),
               "days_since_mark": int((t1 - mdays[i_last]).days)}
        scored_any = False
        for W in WINDOWS:
            start = t1 - pd.Timedelta(days=W)
            i_first = int(mdays.searchsorted(start, side="right"))   # first mark strictly after start
            n_marks = i_last - i_first + 1
            # The vault must have existed before the window (a mark at or before its start), so a
            # W-day score is never computed on a vault younger than W days.
            if n_marks < MIN_EVENTS + 1 or i_first == 0:
                for f in TRIM_FRACTIONS:
                    tag = f"f{int(f * 100):02d}"
                    row[f"ret{W}_{tag}"] = np.nan; row[f"sharpe{W}_{tag}"] = np.nan
                row[f"sortino{W}"] = np.nan; row[f"vol{W}"] = np.nan; row[f"events{W}"] = int(max(n_marks, 0))
                continue
            # Event returns between consecutive marks INSIDE the window only: the first event starts
            # at the first mark in the window, so no return interval begins before the window.
            seg = prices[i_first:i_last + 1]
            r = np.diff(np.log(seg))
            s = event_scores(r, W)
            for key, value in s.items():
                base, _, tag = key.partition("_")
                row[f"{base}{W}_{tag}" if tag else f"{base}{W}"] = value
            scored_any = True
        if not scored_any:
            dropped["stale_or_too_few_marks"] += 1
            continue
        i_carry = int(mdays.searchsorted(t, side="right")) - 1
        carried = float(prices[i_carry])
        ok_any = False
        for H, min_marks in HORIZONS.items():
            t_end = t + pd.Timedelta(days=H)
            j0 = int(mdays.searchsorted(t, side="right")); j1 = int(mdays.searchsorted(t_end, side="right"))
            fwd = prices[j0:j1]
            if len(fwd) < min_marks or (t_end - mdays[j1 - 1]).days > STALE_DAYS:
                for k in ("return", "vol", "sharpe", "log_max_dd"):
                    row[f"fwd{H}_{k}"] = np.nan
                row[f"fwd{H}_events"] = int(len(fwd))
                continue
            row.update(forward_outcome(carried, fwd, H))
            ok_any = True
        if not ok_any:
            dropped["forward_marks"] += 1
            continue
        rows.append(row)
panel = pd.DataFrame(rows)
panel["young"] = panel["age_days"] < YOUNG_DAYS
panel["regime_contained"] = [regime_contained(t, PRIMARY_H) for t in panel["date"]]
print("candidate-dates dropped:", dropped)
print(f"panel: {len(panel):,} rows, {panel['address'].nunique()} vaults, {panel['date'].nunique()} decisions "
      f"{panel['date'].min().date()} to {panel['date'].max().date()}; young (< {YOUNG_DAYS} d) share of rows {panel['young'].mean():.1%}")
by_regime = panel.groupby("regime").agg(rows=("address", "size"), vaults=("address", "nunique"), decisions=("date", "nunique"),
                                        events90_median=("events90", "median"), events180_median=("events180", "median"),
                                        fwd60_events_median=("fwd60_events", "median"), fwd30_events_median=("fwd30_events", "median"),
                                        fwd60_finite=("fwd60_sharpe", lambda s: float(np.isfinite(s).mean())),
                                        fwd30_finite=("fwd30_sharpe", lambda s: float(np.isfinite(s).mean())),
                                        young_share=("young", "mean"))
display(by_regime.round(3))

SIGNALS = []
for W in WINDOWS:
    for f in TRIM_FRACTIONS:
        tag = f"f{int(f * 100):02d}"
        SIGNALS.append({"name": f"ret{W}_{tag}", "direction": "high", "window": W, "trim": f, "family": "return"})
    for f in TRIM_FRACTIONS:
        tag = f"f{int(f * 100):02d}"
        SIGNALS.append({"name": f"sharpe{W}_{tag}", "direction": "high", "window": W, "trim": f, "family": "sharpe"})
    SIGNALS.append({"name": f"sortino{W}", "direction": "high", "window": W, "trim": None, "family": "sortino"})
    SIGNALS.append({"name": f"vol{W}", "direction": "low", "window": W, "trim": None, "family": "vol"})
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
TARGETS = ["fwd60_sharpe", "fwd60_return", "fwd60_vol", "fwd60_log_max_dd", "fwd30_sharpe", "fwd30_return"]
TARGET_SIGN = {"fwd60_sharpe": 1.0, "fwd60_return": 1.0, "fwd60_vol": -1.0, "fwd60_log_max_dd": 1.0, "fwd30_sharpe": 1.0, "fwd30_return": 1.0}
PRIMARY = f"fwd{PRIMARY_H}_sharpe"
coverage = pd.DataFrame({s: np.isfinite(panel[s]).mean() for s in SIGNAL_NAMES}, index=["finite_share"]).T
display(coverage.round(3).T)


## Part 2. One shared bootstrap, every hypothesis

As NB38: per-date signed Spearman, equal-weight mean over dates, one two-way cluster bootstrap
shared by every signal, target and paired difference; simultaneous max-T lower bounds over the
signal family on the primary target and over the paired-comparison family.


In [ ]:
def per_date_blocks(frame: pd.DataFrame) -> dict:
    out = {}
    cols = SIGNAL_NAMES + TARGETS
    for date, g in frame.groupby("date"):
        out[pd.Timestamp(date)] = {"vault": g["address"].to_numpy(), "values": g[cols].to_numpy(dtype=float)}
    return out


def date_statistics(values: np.ndarray) -> np.ndarray:
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    out = np.full((S, T), np.nan)
    targets = values[:, S:]
    for i, name in enumerate(SIGNAL_NAMES):
        x = values[:, i]
        for j, target in enumerate(TARGETS):
            y = targets[:, j]
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < MIN_CANDIDATES:
                continue
            xr, yr = rankdata(x[ok]), rankdata(y[ok])
            if np.ptp(xr) == 0 or np.ptp(yr) == 0:
                continue
            xc, yc = xr - xr.mean(), yr - yr.mean()
            out[i, j] = SIGNAL_SIGN[name] * TARGET_SIGN[target] * float((xc * yc).sum() / math.sqrt((xc ** 2).sum() * (yc ** 2).sum()))
    return out


def mean_over_dates(blocks: dict, dates: list, counts: dict | None = None) -> tuple:
    S, T = len(SIGNAL_NAMES), len(TARGETS)
    total, n = np.zeros((S, T)), np.zeros((S, T))
    for date in dates:
        block = blocks.get(date)
        if block is None:
            continue
        values = block["values"]
        if counts is not None:
            repeats = np.array([counts.get(v, 0) for v in block["vault"]], dtype=int)
            if repeats.sum() < MIN_CANDIDATES:
                continue
            values = np.repeat(values, repeats, axis=0)
        stats = date_statistics(values)
        finite = np.isfinite(stats)
        total[finite] += stats[finite]
        n[finite] += 1
    with np.errstate(invalid="ignore"):
        return np.where(n > 0, total / np.maximum(n, 1), np.nan), n


def bootstrap(frame: pd.DataFrame, draws: int = DRAWS, seed: int = SEED, verbose: bool = True, block: int = DATE_BLOCK) -> dict:
    """Two-way cluster bootstrap: NON-WRAPPING moving date blocks of `block` decisions (starts drawn
    uniformly from the positions where a whole block fits) and vault clusters, together."""
    blocks = per_date_blocks(frame)
    dates = sorted(blocks)
    vaults = sorted(frame["address"].unique())
    observed, n_dates = mean_over_dates(blocks, dates)
    rng = np.random.default_rng(seed)
    block = min(block, len(dates))
    n_blocks = int(math.ceil(len(dates) / block))
    reps = np.full((draws,) + observed.shape, np.nan)
    for d in range(draws):
        starts = rng.integers(0, len(dates) - block + 1, size=n_blocks)
        index = np.concatenate([np.arange(s, s + block) for s in starts])[:len(dates)]
        drawn = rng.choice(len(vaults), size=len(vaults), replace=True)
        counts = {}
        for v in drawn:
            counts[vaults[v]] = counts.get(vaults[v], 0) + 1
        reps[d], _ = mean_over_dates(blocks, [dates[i] for i in index], counts)
        if verbose and (d + 1) % 100 == 0:
            print(f"  draw {d + 1}/{draws}")
    return {"observed": observed, "draws": reps, "n_dates": n_dates, "dates": dates, "rows": len(frame)}


def simultaneous_lower(observed: np.ndarray, reps: np.ndarray, level: float = LEVEL) -> dict:
    se = np.nanstd(reps, axis=0, ddof=1)
    member = np.isfinite(observed) & np.isfinite(se) & (se > 0)
    stud = (reps - observed[None, :]) / np.where(se > 0, se, np.nan)[None, :]
    complete = np.isfinite(stud[:, member]).all(axis=1) if member.any() else np.zeros(len(reps), dtype=bool)
    per_draw_max = stud[complete][:, member].max(axis=1) if complete.any() else np.array([])
    critical = float(np.percentile(per_draw_max, level * 100.0)) if len(per_draw_max) >= 100 else float("nan")
    lower = np.where(member, observed - critical * se, np.nan)
    centred = reps - observed[None, :]
    p = (1.0 + (centred >= observed[None, :]).sum(axis=0)) / (len(reps) + 1.0)
    return {"se": se, "critical": critical, "lower": lower, "lower_unadjusted": np.nanpercentile(reps, (1 - level) * 100.0, axis=0),
            "p": p, "family_size": int(member.sum()), "complete_draws": int(len(per_draw_max))}


def screen(frame: pd.DataFrame, label: str, verbose: bool = True, block: int = DATE_BLOCK) -> dict:
    print(f"{label}: {len(frame):,} rows, {frame['date'].nunique()} decisions, {frame['address'].nunique()} vaults, date block {block}")
    boot = bootstrap(frame, verbose=verbose, block=block)
    j = TARGETS.index(PRIMARY)
    fam = simultaneous_lower(boot["observed"][:, j], boot["draws"][:, :, j])
    rows = []
    for i, s in enumerate(SIGNALS):
        row = {"signal": s["name"], "family": s["family"], "window": s["window"], "trim": s["trim"],
               "dates": int(boot["n_dates"][i, j]), "evaluated": bool(boot["n_dates"][i, j] >= MIN_DATES and fam["se"][i] > 0)}
        for t_idx, target in enumerate(TARGETS):
            row[f"rho_{target}"] = boot["observed"][i, t_idx]
        row["se_primary"] = fam["se"][i]
        row["lo_primary_simultaneous"] = fam["lower"][i]
        row["lo_primary_unadjusted"] = fam["lower_unadjusted"][i]
        row["p_primary"] = fam["p"][i]
        rows.append(row)
    table = pd.DataFrame(rows).set_index("signal")
    diffs, d_obs, d_reps = [], [], []
    for W in WINDOWS:
        for fam_name in ("ret", "sharpe"):
            base = SIGNAL_NAMES.index(f"{fam_name}{W}_f00")
            for f in TRIM_FRACTIONS[1:]:
                idx = SIGNAL_NAMES.index(f"{fam_name}{W}_f{int(f * 100):02d}")
                obs = boot["observed"][idx, j] - boot["observed"][base, j]
                rep = boot["draws"][:, idx, j] - boot["draws"][:, base, j]
                d_obs.append(obs); d_reps.append(rep)
                fin = rep[np.isfinite(rep)]; n = len(fin); centred = fin - obs
                p_hi = (1.0 + (centred >= obs).sum()) / (n + 1.0); p_lo = (1.0 + (centred <= obs).sum()) / (n + 1.0)
                diffs.append({"family": fam_name, "window": W, "trim": f, "trimmed_rho": boot["observed"][idx, j],
                              "raw_rho": boot["observed"][base, j], "difference": obs,
                              "ci_lo": float(np.percentile(fin, 2.5)) if n >= 100 else np.nan,
                              "ci_hi": float(np.percentile(fin, 97.5)) if n >= 100 else np.nan,
                              "p_two_sided_add_one": float(min(1.0, 2 * min(p_hi, p_lo))) if n >= 100 else np.nan, "draws": int(n)})
    paired = pd.DataFrame(diffs)
    pfam = simultaneous_lower(np.array(d_obs), np.column_stack(d_reps))
    paired["lo_simultaneous_family"] = pfam["lower"]
    paired["se"] = pfam["se"]
    return {"label": label, "table": table, "paired": paired, "critical": fam["critical"], "family_size": fam["family_size"],
            "complete_draws": fam["complete_draws"], "boot": boot, "paired_critical": pfam["critical"], "paired_family_size": pfam["family_size"],
            "block": block}


TABLE_COLS = ["family", "window", "trim", "dates", "evaluated", f"rho_{PRIMARY}", "se_primary", "lo_primary_simultaneous",
              "lo_primary_unadjusted", "p_primary", "rho_fwd60_return", "rho_fwd60_vol", "rho_fwd60_log_max_dd", "rho_fwd30_sharpe", "rho_fwd30_return"]
full = screen(panel, "all regimes")
print(f"\nprimary family: {full['family_size']} signals, critical {full['critical']:.4f} on {full['complete_draws']} complete draws")
display(full["table"][TABLE_COLS].round(4))
# Sensitivity to the block length: 45 decisions (90 days), the trailing-score persistence length.
full45 = screen(panel, "all regimes, block 45", verbose=False, block=DATE_BLOCK_SENSITIVITY)
full90 = screen(panel, "all regimes, block 90", verbose=False, block=DATE_BLOCK_LONG)
sens = pd.DataFrame({"rho": full["table"][f"rho_{PRIMARY}"], "lo_block30": full["table"]["lo_primary_simultaneous"],
                     "lo_block45": full45["table"]["lo_primary_simultaneous"], "lo_block90": full90["table"]["lo_primary_simultaneous"],
                     "se_block30": full["table"]["se_primary"], "se_block45": full45["table"]["se_primary"], "se_block90": full90["table"]["se_primary"]})
for b in (30, 45, 90):
    sens[f"clears_block{b}"] = sens[f"lo_block{b}"] > 0
print(f"\nblock-length sensitivity (critical {full['critical']:.3f} at 30, {full45['critical']:.3f} at 45, {full90['critical']:.3f} at 90 decisions): "
      f"{int(sens['clears_block30'].sum())} signals clear at block 30, {int(sens['clears_block45'].sum())} at 45, {int(sens['clears_block90'].sum())} at 90 "
      f"(90 decisions = 180 days = the longest trailing window; {math.ceil(len(full['boot']['dates']) / DATE_BLOCK_LONG)} blocks per draw)")
display(sens.round(4))
print(f"\nPAIRED trimmed - raw on {PRIMARY} (per-comparison 95% intervals; simultaneous lower bound over the "
      f"{full['paired_family_size']} paired comparisons, critical {full['paired_critical']:.4f}):")
display(full["paired"].round(4))


### Reachability

A noisy foresight oracle (the primary target plus 5% noise) through the identical machinery must
clear the family-wise lower bound; otherwise an all-fail result says nothing about the signals.


In [ ]:
rng = np.random.default_rng(SEED + 1)
oracle_panel = panel.copy()
oracle_panel["oracle"] = oracle_panel[PRIMARY] + rng.normal(0.0, 0.05 * float(np.nanstd(oracle_panel[PRIMARY])), len(oracle_panel))
_saved = (SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN)
SIGNALS = list(SIGNALS) + [{"name": "oracle", "direction": "high", "window": None, "trim": None, "family": "oracle"}]
SIGNAL_NAMES = [s["name"] for s in SIGNALS]
SIGNAL_SIGN = {s["name"]: (1.0 if s["direction"] == "high" else -1.0) for s in SIGNALS}
try:
    oracle_res = screen(oracle_panel, "oracle reachability", verbose=False)
finally:
    SIGNALS, SIGNAL_NAMES, SIGNAL_SIGN = _saved
orow = oracle_res["table"].loc["oracle"]
print(f"oracle: rho {orow[f'rho_{PRIMARY}']:.4f}, simultaneous lower bound {orow['lo_primary_simultaneous']:.4f} over "
      f"{oracle_res['family_size']} signals (critical {oracle_res['critical']:.4f}); finite primary target rows "
      f"{int(np.isfinite(panel[PRIMARY]).sum())} of {len(panel)}")
assert orow["lo_primary_simultaneous"] > 0, "the screen cannot produce a positive simultaneous bound even for a foresight oracle"
print("reachable")


## Part 3. Per regime, and young against old

A regime cohort is the set of decisions whose decision date AND whole 60-day forward horizon lie
inside the regime, so a weekly-regime outcome is measured on weekly marks. Each cohort is a
separate sample with a separate bootstrap. Young (< 360 days) and old are split on the whole
panel; they have different date coverage and are estimates on different samples, not a test of
a difference.


In [ ]:
SHORT_COLS = ["window", "trim", "dates", "evaluated", f"rho_{PRIMARY}", "lo_primary_simultaneous", "p_primary",
              "rho_fwd60_return", "rho_fwd60_vol", "rho_fwd30_sharpe"]
by_regime_screens = {}
print("decisions whose whole 60-day horizon lies inside one regime:", panel.groupby("regime_contained")["date"].nunique().to_dict())
for name, a, b in REGIMES:
    sub = panel[panel["regime_contained"] == name]
    if sub["date"].nunique() < MIN_DATES:
        print(f"{name}: only {sub['date'].nunique()} decisions with the whole horizon inside the regime - not screened")
        continue
    res = screen(sub, name, verbose=False)
    by_regime_screens[name] = res
    print(f"  family {res['family_size']}, critical {res['critical']:.4f}, complete draws {res['complete_draws']}")
    display(res["table"][SHORT_COLS].round(4))
    print(f"  paired trimmed - raw on {PRIMARY} (family critical {res['paired_critical']:.4f}):")
    display(res["paired"].round(4))
young = screen(panel[panel["young"]], "young (< 360 days)", verbose=False)
old = screen(panel[~panel["young"]], "old (>= 360 days)", verbose=False)
for res in (young, old):
    print(f"\n{res['label']}: family {res['family_size']}, critical {res['critical']:.4f}")
    display(res["table"][SHORT_COLS].round(4))
    display(res["paired"].round(4))


## Part 4. Manifest


In [ ]:
def table_records(res):
    return {"table": res["table"].round(6).to_dict(orient="index"), "paired": res["paired"].round(6).to_dict(orient="records"),
            "critical": res["critical"], "family_size": res["family_size"], "complete_draws": res["complete_draws"],
            "paired_critical": res["paired_critical"], "paired_family_size": res["paired_family_size"],
            "rows": int(res["boot"]["rows"]), "decisions": int(len(res["boot"]["dates"]))}

manifest = {
    "verdict": "DIAGNOSTIC - a vault-level screen on the full archive, not a result",
    "provenance": {**PROVENANCE, "last_mark": str(LAST_MARK)},
    "constants": {"windows": list(WINDOWS), "trim_fractions": list(TRIM_FRACTIONS), "horizons": {str(k): v for k, v in HORIZONS.items()},
                  "primary": PRIMARY, "panel_start": str(PANEL_START.date()), "min_tvl_usd": MIN_TVL_USD, "min_events": MIN_EVENTS,
                  "stale_days": STALE_DAYS, "min_candidates": MIN_CANDIDATES, "min_dates": MIN_DATES, "young_days": YOUNG_DAYS,
                  "draws": DRAWS, "date_block": DATE_BLOCK, "date_block_sensitivity": DATE_BLOCK_SENSITIVITY, "date_block_long": DATE_BLOCK_LONG, "seed": SEED},
    "regimes": [(n, str(a.date()), str(b.date())) for n, a, b in REGIMES],
    "density": dens.round(6).reset_index().astype({"month": str}).to_dict(orient="records"),
    "panel": {"rows": int(len(panel)), "vaults": int(panel["address"].nunique()), "decisions": int(panel["date"].nunique()),
              "first": str(panel["date"].min().date()), "last": str(panel["date"].max().date()), "young_share": float(panel["young"].mean())},
    "dropped": dropped,
    "by_regime": by_regime.round(6).to_dict(orient="index"),
    "coverage": coverage["finite_share"].round(6).to_dict(),
    "screens": {"all": table_records(full), "all_block45": table_records(full45), "all_block90": table_records(full90),
                **{n: table_records(r) for n, r in by_regime_screens.items()},
                "young": table_records(young), "old": table_records(old)},
    "block_sensitivity": sens.round(6).to_dict(orient="index"),
    "regime_contained_decisions": {k: int(v) for k, v in panel.groupby("regime_contained")["date"].nunique().to_dict().items()},
    "oracle": {"rho": float(orow[f"rho_{PRIMARY}"]), "lo_simultaneous": float(orow["lo_primary_simultaneous"]),
               "family_size": oracle_res["family_size"], "critical": oracle_res["critical"]},
}
Path("_build/manifest_39.json").write_text(json.dumps(manifest, indent=1, default=str))
print("wrote _build/manifest_39.json")
